# Avalanche Detection & Power-Law Analysis

Implements the **Beggs & Plenz (2003)** avalanche detection method on synthetic
branching process spike trains at three criticality regimes:

- **Subcritical** (σ_b = 0.6): exponential size distribution
- **Critical** (σ_b = 1.0): power-law P(S) ~ S^(-3/2)
- **Supercritical** (σ_b = 1.4): bimodal / runaway

Power-law exponent τ is fitted via MLE (Clauset et al. 2009).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from neural_avalanche_utac.spike_train import SpikeTrainConfig, SpikeTrainGenerator
from neural_avalanche_utac.avalanche import AvalancheDetector
from neural_avalanche_utac.power_law import PowerLawFitter

In [ ]:
# Generate spike trains at three regimes
regimes = {'subcritical': (0.6, 43), 'critical': (1.0, 42), 'supercritical': (1.4, 44)}
spikes_dict = {}

for label, (sigma_b, seed) in regimes.items():
    cfg = SpikeTrainConfig(n_neurons=300, duration_s=300.0, dt_ms=2.0,
                           branching_ratio=sigma_b, seed=seed)
    spikes_dict[label] = SpikeTrainGenerator(cfg).generate()['spikes']
    print(f'{label:15s} σ_b={sigma_b}  shape={spikes_dict[label].shape}')

In [ ]:
# Detect avalanches and fit power laws
det = AvalancheDetector()
fitter = PowerLawFitter(x_min=1.0)

results = {}
for label, spikes in spikes_dict.items():
    avs = det.detect(spikes)
    sizes = det.sizes(avs)
    fit = fitter.fit_and_score(sizes, tau_critical=1.5) if len(avs) > 10 else {}
    results[label] = {'avs': avs, 'sizes': sizes, 'fit': fit}
    tau = fit.get('tau', float('nan'))
    prox = fit.get('tau_proximity', 0.0)
    print(f'{label:15s}  n_avalanches={len(avs):5d}  tau={tau:.3f}  R(proximity)={prox:.3f}')

In [ ]:
# Plot avalanche size distributions
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
colors = {'subcritical': 'steelblue', 'critical': 'crimson', 'supercritical': 'forestgreen'}

x_ref = np.logspace(0, 3, 100)

for ax, (label, data) in zip(axes, results.items()):
    sizes = data['sizes']
    if len(sizes) == 0:
        continue
    bins = np.logspace(0, np.log10(sizes.max() + 1), 30)
    ax.hist(sizes, bins=bins, color=colors[label], alpha=0.7, density=True)
    if data['fit']:
        tau = data['fit']['tau']
        y_ref = x_ref ** (-tau)
        y_ref /= y_ref[0]
        ax.plot(x_ref, y_ref * 0.1, 'k--', lw=1.5, label=f'τ={tau:.2f}')
    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_title(f'{label.capitalize()}\nτ_critical=1.5')
    ax.set_xlabel('Avalanche size S'); ax.set_ylabel('P(S)')
    ax.legend()

plt.suptitle('Avalanche Size Distributions — Subcritical / Critical / Supercritical', fontsize=12)
plt.tight_layout()
plt.show()